# Step 4: Knowledge Graph Construction

## Phase 2: Knowledge Graph Construction (continued)

**Goal:** Build course transition graph with concept overlap and temporal precedence

**Graph Structure:**
- **Nodes:** Courses with concept sets
- **Edges:** Weighted by concept_overlap + temporal_precedence

**Edge Weight Formula:**
```
weight(C₁ → C₂) = α·concept_overlap(C₁, C₂) + β·temporal_precedence(C₁, C₂)
```

Where:
- concept_overlap = |concepts(C₁) ∩ concepts(C₂)| / |concepts(C₁) ∪ concepts(C₂)|
- temporal_precedence = P(enroll_C₁ < enroll_C₂ | both taken)

In [1]:
import json
import pickle
import networkx as nx
from collections import defaultdict
import matplotlib.pyplot as plt
import glob

OUTPUT_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/output"

# Weights for edge calculation
ALPHA = 0.6  # Concept overlap weight
BETA = 0.4   # Temporal precedence weight

print("Step 4: Knowledge Graph Construction")
print(f"Weights: α={ALPHA} (concept), β={BETA} (temporal)")

Step 4: Knowledge Graph Construction
Weights: α=0.6 (concept), β=0.4 (temporal)


## Load Data

In [2]:
# Load user sequences
with open(f"{OUTPUT_PATH}/user_sequences.json", "r") as f:
    user_sequences = json.load(f)

# Load course concepts
with open(f"{OUTPUT_PATH}/course_concepts.json", "r") as f:
    course_concepts = json.load(f)

print(f"Loaded {len(user_sequences)} user sequences")
print(f"Loaded {len(course_concepts)} course concept mappings")

Loaded 11385 user sequences
Loaded 887 course concept mappings


## Calculate Course Transition Statistics

In [3]:
# Count temporal transitions between courses
course_transitions = defaultdict(lambda: defaultdict(int))
course_cooccurrence = defaultdict(lambda: defaultdict(int))

print("Counting course transitions...")

for record in user_sequences:
    user_id = record["user_id"]
    sequence = record["sequence"]
    sequence = sorted(sequence, key=lambda x: x[0])
    
    courses = [item[1] for item in sequence]
    
    # Count all pairs
    n = len(courses)
    for i in range(n):
        for j in range(i + 1, n):
            course_a = courses[i]
            course_b = courses[j]
            
            # A comes before B
            course_transitions[course_a][course_b] += 1
            course_cooccurrence[course_a][course_b] += 1
            course_cooccurrence[course_b][course_a] += 1

print(f"Unique course pairs (transitions): {sum(len(v) for v in course_transitions.values())}")

Counting course transitions...
Unique course pairs (transitions): 246332


## Calculate Concept Overlap

In [4]:
def jaccard_similarity(set_a, set_b):
    """Calculate Jaccard similarity between two sets"""
    if not set_a or not set_b:
        return 0.0
    intersection = len(set(set_a) & set(set_b))
    union = len(set(set_a) | set(set_b))
    return intersection / union if union > 0 else 0.0

# Calculate concept overlap for course pairs
concept_overlap = {}

print("Calculating concept overlap...")

for course_a, inner_dict in course_transitions.items():
    concept_overlap[course_a] = {}
    concepts_a = course_concepts.get(course_a, [])
    
    for course_b in inner_dict.keys():
        concepts_b = course_concepts.get(course_b, [])
        overlap = jaccard_similarity(concepts_a, concepts_b)
        concept_overlap[course_a][course_b] = overlap

print(f"Calculated concept overlap for {len(concept_overlap)} courses")

Calculating concept overlap...
Calculated concept overlap for 742 courses


## Build Knowledge Graph

In [33]:
# Create directed graph
G = nx.DiGraph()

# Add nodes (courses) with concept attributes
for course_id, concepts in course_concepts.items():
    G.add_node(
        course_id,
        num_concepts=len(concepts),
        concepts=concepts
    )

print(f"Added {G.number_of_nodes()} course nodes")

# Add edges with weights
edge_count = 0
for course_a, inner_dict in course_transitions.items():
    for course_b, transition_count in inner_dict.items():
        # Temporal precedence probability
        temporal_prob = transition_count / course_cooccurrence[course_a][course_b]
        
        # Concept overlap
        overlap = concept_overlap.get(course_a, {}).get(course_b, 0)
        
        # Combined weight
        weight = ALPHA * overlap + BETA * temporal_prob
        
        # Only add edge if weight > threshold
        if weight > 0.4:
            G.add_edge(
                course_a, course_b,
                weight=weight,
                concept_overlap=overlap,
                temporal_prob=temporal_prob,
                transition_count=transition_count
            )
            edge_count += 1

print(f"Added {edge_count} edges")

Added 887 course nodes
Added 3407 edges


## Graph Analysis

In [34]:
# Graph statistics
print("\nKnowledge Graph Statistics:")
print("="*50)
print(f"Nodes (courses): {G.number_of_nodes()}")
print(f"Edges (transitions): {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")

# Degree statistics
in_degrees = [d for n, d in G.in_degree()]
out_degrees = [d for n, d in G.out_degree()]

print(f"\nIn-degree: avg={sum(in_degrees)/len(in_degrees):.2f}, max={max(in_degrees)}")
print(f"Out-degree: avg={sum(out_degrees)/len(out_degrees):.2f}, max={max(out_degrees)}")

# Top courses by out-degree (prerequisites for many)
top_prereqs = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 prerequisite courses (by out-degree):")
for course, degree in top_prereqs:
    print(f"  {course}: {degree} successors")


Knowledge Graph Statistics:
Nodes (courses): 887
Edges (transitions): 3407
Density: 0.0043

In-degree: avg=3.84, max=43
Out-degree: avg=3.84, max=61

Top 10 prerequisite courses (by out-degree):
  C_769293: 61 successors
  C_655850: 48 successors
  C_707173: 33 successors
  C_682129: 33 successors
  C_681745: 29 successors
  C_707379: 27 successors
  C_682520: 27 successors
  C_682637: 25 successors
  C_697091: 25 successors
  C_681357: 24 successors


## Visualize Subgraph

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import numpy as np

weights = [d["weight"] for _, _, d in G.edges(data=True)]

# =========================
# 1. PREPARE GRAPH
# =========================
top_courses = set([course for course, _ in top_prereqs[:15]])
G_filtered = G.subgraph(top_courses).copy()

weights_all = [d["weight"] for _, _, d in G_filtered.edges(data=True)]
THRESHOLD = 0.427

TOP_K_PER_NODE = 2
G_sub = nx.DiGraph()

# =========================
# 2. FILTER EDGES (CORE FIX)
# =========================
for node in G_filtered.nodes():
    edges = sorted(
        G_filtered.out_edges(node, data=True),
        key=lambda x: x[2].get("weight", 0),
        reverse=True
    )[:TOP_K_PER_NODE]

    for u, v, d in edges:
        if d.get("weight", 0) >= THRESHOLD:
            G_sub.add_edge(u, v, weight=d["weight"])

# =========================
# 3. HANDLE CYCLES (AFTER BUILD)
# =========================
if not nx.is_directed_acyclic_graph(G_sub):
    G_sub = nx.DiGraph(nx.maximum_spanning_tree(G_sub.to_undirected()))

# =========================
# 2. BUILD TRUE HIERARCHY (TOPOLOGICAL LEVEL)
# =========================
levels = {}
for node in nx.topological_sort(G_sub):
    preds = list(G_sub.predecessors(node))
    levels[node] = 0 if not preds else max(levels[p] for p in preds) + 1

# =========================
# 3. LAYERED LAYOUT (CRITICAL)
# =========================
layer_nodes = defaultdict(list)
for node, lvl in levels.items():
    layer_nodes[lvl].append(node)

pos = {}
for lvl, nodes in layer_nodes.items():
    for i, node in enumerate(nodes):
        pos[node] = (i - len(nodes)/2, -lvl * 1.5)
# =========================
# 4. NODE COLOR (ROLE-BASED)
# =========================
max_level = max(levels.values()) if levels else 1

node_colors = []
node_sizes = []

for node in G_sub.nodes():
    lvl = levels[node]

    if lvl == 0:
        node_colors.append("#2ca25f")   # foundation (green)
        node_sizes.append(1400)
    elif lvl >= max_level - 1:
        node_colors.append("#756bb1")   # advanced (purple)
        node_sizes.append(1400)
    else:
        node_colors.append("#fd8d3c")   # intermediate (orange)
        node_sizes.append(1100)

# =========================
# 5. EDGE STYLING (STRONG CONTRAST)
# =========================
edges = list(G_sub.edges())
weights = [G_sub[u][v]["weight"] for u, v in edges]
w_min, w_max = min(weights), max(weights)

edge_widths = [
    3 + 5 * (w - w_min) / (w_max - w_min + 1e-6)
    for w in weights
]
edge_colors = weights      
# =========================
# 6. DRAW GRAPH
# =========================
plt.figure(figsize=(12, 9))

nx.draw_networkx_nodes(
    G_sub, pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors='black',
    linewidths=1.2
)

nx.draw_networkx_edges(
    G_sub, pos,
    edgelist=edges,
    width=edge_widths,
    edge_color=edge_colors,
    edge_cmap=plt.cm.Blues,   # vẫn giữ tông xanh
    alpha=0.9,
    arrows=True,
    arrowsize=20
)

nx.draw_networkx_labels(
    G_sub, pos,
    font_size=9,
    font_weight='bold'
)

plt.title(
    "Course-Level Knowledge Graph",
    fontsize=14
)

plt.axis('off')
plt.tight_layout()

plt.savefig(
    f"{OUTPUT_PATH}/knowledge_graph_final.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

## Save Graph

In [36]:
# Save as GML
nx.write_gml(G, f"{OUTPUT_PATH}/knowledge_graph.gml")

# Save as pickle for Python loading
with open(f"{OUTPUT_PATH}/knowledge_graph.pkl", "wb") as f:
    pickle.dump(G, f)

# Also save edge list as CSV for inspection
edge_list = []
for u, v, data in G.edges(data=True):
    edge_list.append({
        'source': u,
        'target': v,
        'weight': data['weight'],
        'concept_overlap': data['concept_overlap'],
        'temporal_prob': data['temporal_prob']
    })

import pandas as pd
edge_df = pd.DataFrame(edge_list)
edge_df.to_csv(f"{OUTPUT_PATH}/knowledge_graph_edges.csv", index=False)

print(f"\nSaved:")
print(f"  - knowledge_graph.gml")
print(f"  - knowledge_graph.pkl")
print(f"  - knowledge_graph_edges.csv")

print("\n" + "="*60)
print("STEP 4 COMPLETE!")
print("="*60)


Saved:
  - knowledge_graph.gml
  - knowledge_graph.pkl
  - knowledge_graph_edges.csv

STEP 4 COMPLETE!
